# TSE Filiacao Playground

Use this notebook to experiment with the TSE party-affiliation data from Base dos Dados and merge it to the municipal BVR treatment-timing file. It is designed for safe iteration:

- BigQuery execution is **off by default**.
- Every query has a dry-run guardrail before execution.
- Query outputs are cached under `data/interim/tse_filiacao/`.
- The local merge uses `data/clean/tse_bvr/municipality_bvr_first_treat.parquet` and the existing TSE municipality panel.

The production pipeline is `src/analysis/build_tse_filiacao_flow_panel.py`. This notebook is for exploration, diagnostics, and specification tinkering.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

CREDENTIALS_PATH = ROOT / "credentials" / "gcp-key.json"
INTERIM_DIR = ROOT / "data" / "interim" / "tse_filiacao"
CLEAN_DIR = ROOT / "data" / "clean" / "tse_filiacao"
TREATMENT_PATH = ROOT / "data" / "clean" / "tse_bvr" / "municipality_bvr_first_treat.parquet"
TSE_PANEL_PATH = ROOT / "data" / "clean" / "tse" / "tse_clean_panel_2000_2018.parquet"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300})

print(f"Project root: {ROOT}")
print(f"Credentials file: {CREDENTIALS_PATH.exists()} -> {CREDENTIALS_PATH.relative_to(ROOT)}")
print(f"Treatment file: {TREATMENT_PATH.exists()} -> {TREATMENT_PATH.relative_to(ROOT)}")
print(f"TSE panel file: {TSE_PANEL_PATH.exists()} -> {TSE_PANEL_PATH.relative_to(ROOT)}")

## 1. Settings

Flip `RUN_BIGQUERY` only when you want to refresh cached Base dos Dados outputs. The cached files currently let you run the notebook without touching BigQuery.

In [ ]:
RUN_BIGQUERY = False
FORCE_REFRESH = False
MAX_QUERY_GB = 15

ELECTION_YEARS = [2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018]
KNOWN_MUNICIPALITIES = {
    "3550308": "Sao Paulo capital",
    "3304557": "Rio de Janeiro capital",
    "5300108": "Brasilia",
    "2927408": "Salvador",
}

PLAYGROUND = {
    "use_duration_filter": True,
    "drop_2000_2002": False,
    "exclude_hybrid": False,
    "exclude_never_treated": False,
    "outcome": "log_new_affiliations",
    "winsorize": True,
    "winsor_low": 0.01,
    "winsor_high": 0.99,
    "event_min": -8,
    "event_max": 8,
    "event_step": 2,
    "reference_event_time": -2,
    "fixed_effects": ["municipality_id", "election_year"],
    "cluster": "municipality_id",
}

PLAYGROUND

## 2. BigQuery Client and Cache Helpers

The code below follows the project Base dos Dados pattern: initialize the BigQuery client from `credentials/gcp-key.json`, dry-run first, refuse single queries above 15 GB, then write results to parquet/CSV caches.

In [ ]:
def make_bq_client():
    from google.cloud import bigquery
    from google.oauth2 import service_account

    with CREDENTIALS_PATH.open() as f:
        creds_info = json.load(f)
    credentials = service_account.Credentials.from_service_account_info(
        creds_info,
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    return bigquery.Client(credentials=credentials, project=credentials.project_id)


def dry_run_query(client, query: str, label: str, max_gb: float = MAX_QUERY_GB) -> int:
    from google.cloud import bigquery

    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    dry = client.query(query, job_config=job_config)
    processed = int(dry.total_bytes_processed or 0)
    print(f"{label}: will process {processed / 1e9:.2f} GB")
    if processed > max_gb * 1_000_000_000:
        raise RuntimeError(f"{label} would process {processed / 1e9:.2f} GB, above the {max_gb:.1f} GB guardrail.")
    return processed


def query_to_cache(client, label: str, query: str, cache_path: Path, force: bool = FORCE_REFRESH) -> pd.DataFrame:
    if cache_path.exists() and not force:
        print(f"{label}: using cached {cache_path.relative_to(ROOT)}")
        return pd.read_parquet(cache_path)
    processed = dry_run_query(client, query, label)
    rows = list(client.query(query).result())
    df = pd.DataFrame([dict(row) for row in rows])
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_path, index=False)
    df.to_csv(cache_path.with_suffix(".csv"), index=False)
    print(f"{label}: saved {len(df):,} rows; dry-run bytes {processed / 1e9:.2f} GB")
    return df


def load_or_query(label: str, query: str, cache_name: str, run_bigquery: bool = RUN_BIGQUERY) -> pd.DataFrame:
    cache_path = INTERIM_DIR / cache_name
    if cache_path.exists() and not FORCE_REFRESH:
        print(f"{label}: loaded cached {cache_path.relative_to(ROOT)}")
        return pd.read_parquet(cache_path)
    if not run_bigquery:
        raise FileNotFoundError(
            f"Missing cache {cache_path.relative_to(ROOT)}. Set RUN_BIGQUERY=True to execute `{label}`."
        )
    client = make_bq_client()
    return query_to_cache(client, label, query, cache_path, force=FORCE_REFRESH)

## 3. Base dos Dados Queries

These are the queries needed to discover the filiação schema, build the deduplicated flow counts, pull population denominators, and perform diagnostics. The core construction uses a deduplicated union of `microdados` and `microdados_antigos`, because the current table alone undercounts historical affiliation starts.

In [ ]:
def build_union_cte(where_clause: str = "") -> str:
    where_sql = f"\n  WHERE {where_clause}" if where_clause else ""
    return f'''
WITH raw AS (
  SELECT 'microdados' AS source_table,
         sigla_partido,
         sigla_uf,
         id_municipio,
         id_municipio_tse,
         titulo_eleitor AS titulo_eleitoral,
         cpf,
         nome,
         situacao_registro,
         motivo_desfiliacao,
         motivo_cancelamento,
         data_filiacao,
         data_desfiliacao,
         data_cancelamento,
         data_exclusao
  FROM `basedosdados.br_tse_filiacao_partidaria.microdados`{where_sql}
  UNION ALL
  SELECT 'microdados_antigos' AS source_table,
         sigla_partido,
         sigla_uf,
         id_municipio,
         id_municipio_tse,
         titulo_eleitoral,
         NULL AS cpf,
         nome,
         situacao_registro,
         NULL AS motivo_desfiliacao,
         motivo_cancelamento,
         data_filiacao,
         data_desfiliacao,
         data_cancelamento,
         NULL AS data_exclusao
  FROM `basedosdados.br_tse_filiacao_partidaria.microdados_antigos`{where_sql}
),
keyed AS (
  SELECT *,
         TO_HEX(SHA256(CONCAT(
           IFNULL(sigla_partido, ''), '|',
           IFNULL(sigla_uf, ''), '|',
           IFNULL(id_municipio, ''), '|',
           IFNULL(titulo_eleitoral, IFNULL(cpf, IFNULL(nome, ''))), '|',
           CAST(data_filiacao AS STRING)
         ))) AS record_key
  FROM raw
),
dedup AS (
  SELECT
    record_key,
    ANY_VALUE(sigla_partido) AS sigla_partido,
    ANY_VALUE(sigla_uf) AS sigla_uf,
    ANY_VALUE(id_municipio) AS id_municipio,
    ANY_VALUE(id_municipio_tse) AS id_municipio_tse,
    ANY_VALUE(data_filiacao) AS data_filiacao,
    MIN(data_desfiliacao) AS data_desfiliacao,
    MIN(data_cancelamento) AS data_cancelamento,
    MIN(data_exclusao) AS data_exclusao,
    STRING_AGG(DISTINCT source_table, ',' ORDER BY source_table) AS source_tables,
    COUNT(*) AS source_record_count
  FROM keyed
  GROUP BY record_key
)
'''


SCHEMA_TABLES_QUERY = '''
SELECT table_id AS table_name, row_count, size_bytes
FROM `basedosdados.br_tse_filiacao_partidaria.__TABLES__`
ORDER BY table_id
'''

def columns_query(table_name: str) -> str:
    return f'''
SELECT column_name, data_type, ordinal_position, is_nullable
FROM `basedosdados.br_tse_filiacao_partidaria.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = '{table_name}'
ORDER BY ordinal_position
'''

def sample_query(table_name: str, limit: int = 1000) -> str:
    return f'''
SELECT *
FROM `basedosdados.br_tse_filiacao_partidaria.{table_name}`
LIMIT {limit}
'''

TOTALS_QUERY = build_union_cte("data_filiacao IS NOT NULL") + '''
SELECT COUNT(*) AS total_records,
       COUNT(DISTINCT id_municipio) AS n_muni,
       MIN(data_filiacao) AS min_date,
       MAX(data_filiacao) AS max_date,
       COUNTIF(data_filiacao BETWEEN '1998-12-01' AND '2018-11-30') AS n_in_flow_window,
       COUNTIF(source_record_count > 1) AS n_keys_with_duplicate_sources
FROM dedup
'''

YEARLY_COVERAGE_QUERY = build_union_cte("data_filiacao IS NOT NULL") + '''
SELECT EXTRACT(YEAR FROM data_filiacao) AS year_filiacao,
       COUNT(*) AS n_affiliations,
       COUNT(DISTINCT id_municipio) AS n_municipalities
FROM dedup
WHERE data_filiacao >= '1996-01-01'
  AND data_filiacao <= '2019-12-31'
GROUP BY year_filiacao
ORDER BY year_filiacao
'''

ID_LENGTH_QUERY = build_union_cte("data_filiacao BETWEEN '1998-12-01' AND '2018-11-30'") + '''
SELECT LENGTH(CAST(id_municipio AS STRING)) AS id_length,
       COUNT(*) AS n
FROM dedup
GROUP BY id_length
ORDER BY id_length
'''

KNOWN_MUNI_QUERY = build_union_cte("data_filiacao BETWEEN '1998-12-01' AND '2018-11-30'") + '''
SELECT id_municipio, COUNT(*) AS n
FROM dedup
WHERE id_municipio IN ('3550308', '3304557', '5300108', '2927408')
GROUP BY id_municipio
ORDER BY id_municipio
'''

MONTHLY_DISAFFILIATIONS_QUERY = build_union_cte("data_filiacao IS NOT NULL") + '''
SELECT DATE_TRUNC(data_desfiliacao, MONTH) AS month_end,
       COUNT(*) AS n_disaffiliations
FROM dedup
WHERE data_desfiliacao IS NOT NULL
  AND data_desfiliacao BETWEEN '1998-01-01' AND '2019-12-31'
GROUP BY month_end
ORDER BY month_end
'''

FLOW_COUNTS_QUERY = build_union_cte("data_filiacao BETWEEN '1998-12-01' AND '2018-11-30'") + '''
, election_windows AS (
  SELECT year AS election_year,
         DATE(year - 2, 12, 1) AS window_start,
         DATE(year, 11, 30) AS window_end
  FROM UNNEST([2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018]) AS year
),
flow_counts AS (
  SELECT
    ew.election_year,
    d.id_municipio,
    ANY_VALUE(d.sigla_uf) AS sigla_uf,
    COUNT(*) AS n_new_affiliations
  FROM election_windows ew
  JOIN dedup d
    ON d.data_filiacao >= ew.window_start
   AND d.data_filiacao <= ew.window_end
  WHERE d.id_municipio IS NOT NULL
    AND (
      COALESCE(d.data_desfiliacao, d.data_cancelamento, d.data_exclusao) IS NULL
      OR DATE_DIFF(COALESCE(d.data_desfiliacao, d.data_cancelamento, d.data_exclusao), d.data_filiacao, MONTH) >= 6
    )
  GROUP BY ew.election_year, d.id_municipio
)
SELECT *
FROM flow_counts
ORDER BY election_year, id_municipio
'''

FLOW_COUNTS_NO_DURATION_QUERY = build_union_cte("data_filiacao BETWEEN '1998-12-01' AND '2018-11-30'") + '''
, election_windows AS (
  SELECT year AS election_year,
         DATE(year - 2, 12, 1) AS window_start,
         DATE(year, 11, 30) AS window_end
  FROM UNNEST([2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018]) AS year
)
SELECT
  ew.election_year,
  d.id_municipio,
  ANY_VALUE(d.sigla_uf) AS sigla_uf,
  COUNT(*) AS n_new_affiliations_no_duration_filter
FROM election_windows ew
JOIN dedup d
  ON d.data_filiacao >= ew.window_start
 AND d.data_filiacao <= ew.window_end
WHERE d.id_municipio IS NOT NULL
GROUP BY ew.election_year, d.id_municipio
ORDER BY election_year, id_municipio
'''

POPULATION_QUERY = '''
SELECT id_municipio, ano, populacao
FROM `basedosdados.br_ibge_populacao.municipio`
WHERE ano BETWEEN 2000 AND 2018
ORDER BY ano, id_municipio
'''

QUERY_SPECS = {
    "schema_tables": ("filiacao_tables", SCHEMA_TABLES_QUERY, "schema_tables.parquet"),
    "schema_columns_microdados": ("microdados_columns", columns_query("microdados"), "schema_columns_microdados.parquet"),
    "schema_columns_microdados_antigos": ("microdados_antigos_columns", columns_query("microdados_antigos"), "schema_columns_microdados_antigos.parquet"),
    "sample_microdados_1000": ("microdados_sample_1000", sample_query("microdados"), "sample_microdados_1000.parquet"),
    "sample_microdados_antigos_1000": ("microdados_antigos_sample_1000", sample_query("microdados_antigos"), "sample_microdados_antigos_1000.parquet"),
    "totals": ("filiacao_union_dedup_totals", TOTALS_QUERY, "union_dedup_totals.parquet"),
    "yearly_coverage": ("filiacao_union_dedup_yearly_coverage", YEARLY_COVERAGE_QUERY, "union_dedup_yearly_coverage.parquet"),
    "id_lengths": ("filiacao_id_length_check", ID_LENGTH_QUERY, "union_dedup_id_lengths.parquet"),
    "known_municipalities": ("filiacao_known_municipality_check", KNOWN_MUNI_QUERY, "known_municipality_counts.parquet"),
    "monthly_disaffiliations": ("filiacao_monthly_disaffiliations", MONTHLY_DISAFFILIATIONS_QUERY, "monthly_disaffiliations.parquet"),
    "flow_counts": ("flow_counts_election_years_duration_filtered", FLOW_COUNTS_QUERY, "flow_counts_election_years.parquet"),
    "flow_counts_no_duration": ("flow_counts_no_duration_filter", FLOW_COUNTS_NO_DURATION_QUERY, "flow_counts_no_duration_filter_election_years.parquet"),
    "population": ("ibge_population_2000_2018", POPULATION_QUERY, "ibge_population_2000_2018.parquet"),
}

list(QUERY_SPECS)

## 4. Load or Refresh Base dos Dados Caches

The default list below loads the cached schema, diagnostics, flow counts, and population data. If a cache is missing, set `RUN_BIGQUERY = True` in the settings cell and rerun.

In [ ]:
QUERY_KEYS_TO_LOAD = [
    "schema_tables",
    "schema_columns_microdados",
    "schema_columns_microdados_antigos",
    "totals",
    "yearly_coverage",
    "id_lengths",
    "known_municipalities",
    "monthly_disaffiliations",
    "flow_counts",
    "flow_counts_no_duration",
    "population",
]

# Optional and relatively expensive because LIMIT does not make BigQuery scan-free.
# Add these keys if you want fresh row samples:
# QUERY_KEYS_TO_LOAD += ["sample_microdados_1000", "sample_microdados_antigos_1000"]

bdd = {}
for key in QUERY_KEYS_TO_LOAD:
    label, query, cache_name = QUERY_SPECS[key]
    bdd[key] = load_or_query(label, query, cache_name)

{key: df.shape for key, df in bdd.items()}

## 5. Schema and Coverage Diagnostics

These quick checks are worth rerunning whenever you refresh from Base dos Dados.

In [ ]:
display(bdd["schema_tables"])

columns = pd.concat([
    bdd["schema_columns_microdados"].assign(table_name="microdados"),
    bdd["schema_columns_microdados_antigos"].assign(table_name="microdados_antigos"),
], ignore_index=True)
display(columns)

display(bdd["totals"])

id_lengths = bdd["id_lengths"].copy()
display(id_lengths)

known = bdd["known_municipalities"].copy()
known["municipality"] = known["id_municipio"].astype(str).map(KNOWN_MUNICIPALITIES)
display(known)

yearly = bdd["yearly_coverage"].copy()
yearly["year_filiacao"] = pd.to_numeric(yearly["year_filiacao"])
yearly["n_affiliations"] = pd.to_numeric(yearly["n_affiliations"])
yearly["n_municipalities"] = pd.to_numeric(yearly["n_municipalities"])
display(yearly)

fig, ax1 = plt.subplots(figsize=(9, 4.6))
ax1.bar(yearly["year_filiacao"], yearly["n_affiliations"] / 1_000_000, color="#294C60", alpha=0.85)
ax1.set_ylabel("Affiliation starts, millions")
ax1.set_xlabel("Affiliation year")
ax2 = ax1.twinx()
ax2.plot(yearly["year_filiacao"], yearly["n_municipalities"], color="#7A4E2D", marker="o", linewidth=1.5)
ax2.set_ylabel("Municipalities with affiliations")
ax1.set_title("Base dos Dados filiation coverage")
ax1.grid(axis="y", linestyle=":", alpha=0.35)
fig.tight_layout()
plt.show()

## 6. Administrative Cleanup Diagnostic

The main flow outcome counts new affiliations, but disaffiliation spikes are useful context. Months above five times the median monthly disaffiliation count are flagged as likely administrative cleanup events.

In [ ]:
monthly = bdd["monthly_disaffiliations"].copy()
monthly["month_end"] = pd.to_datetime(monthly["month_end"])
monthly["n_disaffiliations"] = pd.to_numeric(monthly["n_disaffiliations"])
threshold = 5 * monthly["n_disaffiliations"].median()
monthly["threshold_5x_median"] = threshold
monthly["administrative_event_flag"] = monthly["n_disaffiliations"] > threshold

display(monthly.sort_values("n_disaffiliations", ascending=False).head(25))

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(monthly["month_end"], monthly["n_disaffiliations"], color="#294C60", linewidth=1.2)
ax.axhline(threshold, color="#7A4E2D", linestyle="--", linewidth=1.2, label="5 x median")
flagged = monthly[monthly["administrative_event_flag"]]
ax.scatter(flagged["month_end"], flagged["n_disaffiliations"], color="#7A4E2D", s=18, zorder=3, label="flagged")
ax.set_title("Disaffiliation spikes")
ax.set_xlabel("Month")
ax.set_ylabel("Disaffiliations")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.35)
fig.tight_layout()
plt.show()

## 7. Merge Flow Counts to BVR Treatment Timing

This cell builds the municipality x election-year panel. The key merge is `id_municipio` from Base dos Dados to `municipality_id` in the project files; both are 7-digit IBGE municipality codes.

In [ ]:
def event_grid(config: dict = PLAYGROUND) -> list[int]:
    return list(range(config["event_min"], config["event_max"] + config["event_step"], config["event_step"]))


def event_col(k: int) -> str:
    return f"event_m{abs(k)}" if k < 0 else f"event_p{k}"


def bin_event_time(value: float, config: dict = PLAYGROUND) -> float:
    if pd.isna(value):
        return np.nan
    if value <= config["event_min"]:
        return float(config["event_min"])
    if value >= config["event_max"]:
        return float(config["event_max"])
    return float(config["event_step"] * math.floor(value / config["event_step"]))


def winsorize_series(s: pd.Series, low: float = 0.01, high: float = 0.99) -> pd.Series:
    lo, hi = s.quantile([low, high])
    return s.clip(lo, hi)


def load_flow_data(use_duration_filter: bool = True) -> pd.DataFrame:
    flow = bdd["flow_counts"].copy() if use_duration_filter else bdd["flow_counts_no_duration"].copy()
    flow["id_municipio"] = flow["id_municipio"].astype("string")
    flow["election_year"] = pd.to_numeric(flow["election_year"]).astype(int)
    if "n_new_affiliations" not in flow.columns:
        flow = flow.rename(columns={"n_new_affiliations_no_duration_filter": "n_new_affiliations"})
    flow["n_new_affiliations"] = pd.to_numeric(flow["n_new_affiliations"])
    return flow[["election_year", "id_municipio", "sigla_uf", "n_new_affiliations"]].copy()


def build_filiacao_panel(config: dict = PLAYGROUND) -> pd.DataFrame:
    flow = load_flow_data(config["use_duration_filter"])
    no_filter = bdd["flow_counts_no_duration"].copy()
    no_filter["id_municipio"] = no_filter["id_municipio"].astype("string")
    no_filter["election_year"] = pd.to_numeric(no_filter["election_year"]).astype(int)
    no_filter["n_new_affiliations_no_duration_filter"] = pd.to_numeric(no_filter["n_new_affiliations_no_duration_filter"])

    tse_panel = pd.read_parquet(TSE_PANEL_PATH).copy()
    tse_panel["municipality_id"] = tse_panel["municipality_id"].astype("string")
    muni = (
        tse_panel[["municipality_id", "municipality_name", "state", "hybrid"]]
        .drop_duplicates("municipality_id")
        .sort_values("municipality_id")
    )
    years = pd.DataFrame({"election_year": ELECTION_YEARS})
    grid = muni.assign(_key=1).merge(years.assign(_key=1), on="_key").drop(columns="_key")

    panel = grid.merge(
        flow.rename(columns={"id_municipio": "municipality_id"}),
        on=["municipality_id", "election_year"],
        how="left",
        validate="many_to_one",
    )
    panel["n_new_affiliations"] = panel["n_new_affiliations"].fillna(0).astype(float)
    panel = panel.merge(
        no_filter[["election_year", "id_municipio", "n_new_affiliations_no_duration_filter"]].rename(columns={"id_municipio": "municipality_id"}),
        on=["municipality_id", "election_year"],
        how="left",
        validate="many_to_one",
    )
    panel["n_new_affiliations_no_duration_filter"] = panel["n_new_affiliations_no_duration_filter"].fillna(0).astype(float)

    treatment = pd.read_parquet(TREATMENT_PATH).copy()
    treatment["municipality_id"] = treatment["municipality_id"].astype("string")
    treatment = treatment[["municipality_id", "year_first_treat"]].rename(columns={"year_first_treat": "first_treatment_year"})
    panel = panel.merge(treatment, on="municipality_id", how="left", validate="many_to_one")
    panel["first_treatment_year"] = panel["first_treatment_year"].fillna(9999).astype(int)
    panel["treated"] = panel["first_treatment_year"].lt(9999).astype(int)
    panel["post"] = (panel["election_year"] >= panel["first_treatment_year"]).astype(int)
    panel.loc[panel["first_treatment_year"].ge(9999), "post"] = 0
    panel["event_time"] = panel["election_year"] - panel["first_treatment_year"]
    panel.loc[panel["first_treatment_year"].ge(9999), "event_time"] = np.nan
    panel["event_time_bin"] = panel["event_time"].apply(lambda x: bin_event_time(x, config))
    for k in event_grid(config):
        if k == config["reference_event_time"]:
            continue
        panel[event_col(k)] = panel["event_time_bin"].eq(k).astype(int)

    population = bdd["population"].copy()
    population["id_municipio"] = population["id_municipio"].astype("string")
    population["ano"] = pd.to_numeric(population["ano"]).astype(int)
    population["populacao"] = pd.to_numeric(population["populacao"], errors="coerce")
    panel = panel.merge(
        population.rename(columns={"id_municipio": "municipality_id", "ano": "election_year", "populacao": "population"}),
        on=["municipality_id", "election_year"],
        how="left",
        validate="many_to_one",
    )

    panel["log_new_affiliations"] = np.log1p(panel["n_new_affiliations"])
    panel["log_new_affiliations_no_duration_filter"] = np.log1p(panel["n_new_affiliations_no_duration_filter"])
    panel["new_affiliations_per_pop"] = panel["n_new_affiliations"] / panel["population"] * 1000
    panel["duration_filter_removed"] = panel["n_new_affiliations_no_duration_filter"] - panel["n_new_affiliations"]
    panel["state_year"] = panel["state"].astype(str) + "_" + panel["election_year"].astype(str)

    return panel


panel = build_filiacao_panel(PLAYGROUND)
panel.shape

In [ ]:
summary = pd.Series({
    "rows": len(panel),
    "municipalities": panel["municipality_id"].nunique(),
    "election_years": panel["election_year"].nunique(),
    "treated_municipalities": panel.loc[panel["treated"].eq(1), "municipality_id"].nunique(),
    "never_treated_municipalities": panel.loc[panel["treated"].eq(0), "municipality_id"].nunique(),
    "hybrid_municipalities": panel.loc[pd.to_numeric(panel["hybrid"], errors="coerce").fillna(0).ne(0), "municipality_id"].nunique(),
    "missing_population_rows": int(panel["population"].isna().sum()),
    "positive_flow_rows": int(panel["n_new_affiliations"].gt(0).sum()),
})
display(summary.to_frame("value"))

display(panel[panel["municipality_id"].isin(KNOWN_MUNICIPALITIES)].sort_values(["municipality_id", "election_year"]).head(40))

year_summary = panel.groupby("election_year", as_index=False).agg(
    total_new_affiliations=("n_new_affiliations", "sum"),
    total_new_affiliations_no_duration_filter=("n_new_affiliations_no_duration_filter", "sum"),
    municipalities_with_positive_flow=("n_new_affiliations", lambda s: int((s > 0).sum())),
    population_missing=("population", lambda s: int(s.isna().sum())),
)
display(year_summary)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(year_summary["election_year"], year_summary["total_new_affiliations"] / 1_000_000, marker="o", color="#294C60", label="6-month duration filter")
ax.plot(year_summary["election_year"], year_summary["total_new_affiliations_no_duration_filter"] / 1_000_000, marker="o", color="#7A4E2D", label="No duration filter")
ax.set_title("National two-year flow by election window")
ax.set_xlabel("Election year")
ax.set_ylabel("New affiliations, millions")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.35)
fig.tight_layout()
plt.show()

## 8. Choose an Exploratory Estimation Sample

Edit `PLAYGROUND` above, rebuild `panel`, and rerun this cell. The outcome choices available immediately are `log_new_affiliations`, `log_new_affiliations_no_duration_filter`, and `new_affiliations_per_pop`.

In [ ]:
def make_estimation_sample(panel: pd.DataFrame, config: dict = PLAYGROUND) -> pd.DataFrame:
    df = panel.copy()
    if config["drop_2000_2002"]:
        df = df[~df["election_year"].isin([2000, 2002])].copy()
    if config["exclude_hybrid"]:
        df = df[pd.to_numeric(df["hybrid"], errors="coerce").fillna(0).eq(0)].copy()
    if config["exclude_never_treated"]:
        df = df[df["treated"].eq(1)].copy()
    outcome = config["outcome"]
    df = df.dropna(subset=[outcome]).copy()
    if config["winsorize"]:
        df[outcome + "_w"] = winsorize_series(df[outcome], config["winsor_low"], config["winsor_high"])
    else:
        df[outcome + "_w"] = df[outcome]
    return df


sample = make_estimation_sample(panel, PLAYGROUND)
display(pd.Series({
    "rows": len(sample),
    "municipalities": sample["municipality_id"].nunique(),
    "years": sample["election_year"].nunique(),
    "treated_municipalities": sample.loc[sample["treated"].eq(1), "municipality_id"].nunique(),
    "never_treated_municipalities": sample.loc[sample["treated"].eq(0), "municipality_id"].nunique(),
    "hybrid_municipalities": sample.loc[pd.to_numeric(sample["hybrid"], errors="coerce").fillna(0).ne(0), "municipality_id"].nunique(),
    "outcome_mean": sample[PLAYGROUND["outcome"]].mean(),
    "outcome_sd": sample[PLAYGROUND["outcome"]].std(),
}).to_frame("value"))

display(sample[["municipality_id", "municipality_name", "state", "election_year", "first_treatment_year", "event_time", "event_time_bin", "n_new_affiliations", "population", PLAYGROUND["outcome"]]].head())

## 9. Exploratory TWFE Event Study

This uses within-transformation rather than thousands of explicit municipality dummies. It is meant for quick graph tinkering. For paper outputs, rerun the production estimator pipeline.

In [ ]:
def residualize(df: pd.DataFrame, columns: list[str], groups: list[str], max_iter: int = 200, tol: float = 1e-10) -> pd.DataFrame:
    resid = df[columns].astype(float).copy()
    resid = resid - resid.mean(axis=0)
    for _ in range(max_iter):
        old = resid.to_numpy(copy=True)
        for group in groups:
            means = resid.groupby(df[group], observed=True).transform("mean")
            resid = resid - means
        resid = resid - resid.mean(axis=0)
        if np.nanmax(np.abs(resid.to_numpy() - old)) < tol:
            break
    return resid


def run_twfe_event_study(df: pd.DataFrame, config: dict = PLAYGROUND) -> tuple[pd.DataFrame, object]:
    outcome = config["outcome"] + "_w"
    terms = [event_col(k) for k in event_grid(config) if k != config["reference_event_time"]]
    required = [outcome, config["cluster"]] + config["fixed_effects"] + terms
    work = df.dropna(subset=required).copy()
    cols = [outcome] + terms
    resid = residualize(work, cols, config["fixed_effects"])
    y = resid[outcome]
    X = resid[terms]
    keep = X.var(axis=0).gt(1e-12)
    dropped = list(X.columns[~keep])
    X = X.loc[:, keep]
    result = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": work[config["cluster"]]})

    rows = []
    for k in event_grid(config):
        if k == config["reference_event_time"]:
            rows.append({"event_time": k, "term": "reference", "estimate": 0.0, "std_error": np.nan, "p_value": np.nan, "n_obs": len(work), "dropped": False})
            continue
        term = event_col(k)
        rows.append({
            "event_time": k,
            "term": term,
            "estimate": result.params.get(term, np.nan),
            "std_error": result.bse.get(term, np.nan),
            "p_value": result.pvalues.get(term, np.nan),
            "n_obs": len(work),
            "dropped": term in dropped,
        })
    event_df = pd.DataFrame(rows)
    event_df["ci_low"] = event_df["estimate"] - 1.96 * event_df["std_error"]
    event_df["ci_high"] = event_df["estimate"] + 1.96 * event_df["std_error"]
    return event_df, result


def plot_event_study(event_df: pd.DataFrame, title: str = ""):
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    ci = event_df.dropna(subset=["std_error"])
    ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.75)
    ax.axvline(0, color="0.55", linestyle=":", linewidth=1)
    ax.errorbar(
        ci["event_time"], ci["estimate"], yerr=1.96 * ci["std_error"],
        fmt="o-", color="#294C60", ecolor="#294C60", elinewidth=1.4,
        capsize=3, markersize=5, linewidth=1.8,
    )
    ref = event_df[event_df["term"].eq("reference")]
    if not ref.empty:
        ax.scatter(ref["event_time"], ref["estimate"], color="black", s=28, zorder=4)
    ax.set_xticks(event_grid(PLAYGROUND))
    ax.set_xlabel("Event time")
    ax.set_ylabel("Coefficient")
    ax.set_title(title or PLAYGROUND["outcome"])
    ax.grid(axis="y", linestyle=":", alpha=0.35)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    return fig, ax

In [ ]:
event_df, model = run_twfe_event_study(sample, PLAYGROUND)
display(event_df.round(4))
fig, ax = plot_event_study(
    event_df,
    title=f"{PLAYGROUND['outcome']} | FE: {', '.join(PLAYGROUND['fixed_effects'])}",
)
plt.show()
print(f"N = {int(event_df['n_obs'].max()):,}; clusters = {sample[PLAYGROUND['cluster']].nunique():,}")

## 10. Scenario Comparison

Add or edit scenarios to see how the event-study path changes when you modify data-management choices.

In [ ]:
SCENARIOS = {
    "baseline_log": {},
    "no_duration_filter": {"use_duration_filter": False, "outcome": "log_new_affiliations_no_duration_filter"},
    "per_population": {"outcome": "new_affiliations_per_pop"},
    "drop_2000_2002": {"drop_2000_2002": True},
    "exclude_hybrid": {"exclude_hybrid": True},
    "state_year_fe": {"fixed_effects": ["municipality_id", "state_year"]},
}


def update_config(base: dict, updates: dict) -> dict:
    out = base.copy()
    out.update(updates)
    return out


scenario_rows = []
scenario_summaries = []
for name, updates in SCENARIOS.items():
    cfg = update_config(PLAYGROUND, updates)
    panel_s = build_filiacao_panel(cfg)
    sample_s = make_estimation_sample(panel_s, cfg)
    events_s, _ = run_twfe_event_study(sample_s, cfg)
    events_s["scenario"] = name
    scenario_rows.append(events_s)
    scenario_summaries.append({
        "scenario": name,
        "rows": len(sample_s),
        "municipalities": sample_s["municipality_id"].nunique(),
        "years": sample_s["election_year"].nunique(),
        "outcome": cfg["outcome"],
        "fixed_effects": ", ".join(cfg["fixed_effects"]),
    })

scenario_events = pd.concat(scenario_rows, ignore_index=True)
scenario_summary = pd.DataFrame(scenario_summaries)
display(scenario_summary)

fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.axvline(0, color="0.55", linestyle=":", linewidth=1)
for name, sdf in scenario_events.sort_values(["scenario", "event_time"]).groupby("scenario"):
    ax.plot(sdf["event_time"], sdf["estimate"], marker="o", linewidth=1.6, label=name)
ax.scatter([PLAYGROUND["reference_event_time"]], [0], color="black", s=28, zorder=5)
ax.set_xticks(event_grid(PLAYGROUND))
ax.set_xlabel("Event time")
ax.set_ylabel("Coefficient")
ax.set_title("Filiacao flow scenario comparison")
ax.grid(axis="y", linestyle=":", alpha=0.35)
ax.legend(frameon=False, fontsize=8, ncol=2)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()

## 11. Optional Save

Experimental outputs go under `resources/images/regressions/playground/filiacao/` so they do not overwrite paper figures.

In [ ]:
SAVE_CURRENT_FIGURE = False
SAVE_CURRENT_SAMPLE = False

PLAYGROUND_OUT = ROOT / "resources" / "images" / "regressions" / "playground" / "filiacao"
PLAYGROUND_OUT.mkdir(parents=True, exist_ok=True)

if SAVE_CURRENT_FIGURE:
    out = PLAYGROUND_OUT / f"{PLAYGROUND['outcome']}_event_study_playground.pdf"
    fig, ax = plot_event_study(event_df, title=f"{PLAYGROUND['outcome']} playground")
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out.relative_to(ROOT)}")

if SAVE_CURRENT_SAMPLE:
    out = PLAYGROUND_OUT / f"{PLAYGROUND['outcome']}_sample.csv"
    sample.to_csv(out, index=False)
    print(f"Saved {out.relative_to(ROOT)}")

## Notes for Experimenting

- Keep the six-month duration filter as the default. The no-filter outcome is useful for sensitivity checks because it keeps records that were quickly invalidated.
- The 2000 and 2002 windows are valuable but have thinner source coverage; use `drop_2000_2002=True` as a robustness check.
- Do not normalize by registered voters in this module. BVR mechanically changes the registered-voter denominator.
- The BigQuery flow-count query uses `id_municipio`, a 7-digit IBGE municipality code, and merges directly to the project treatment file.
- This notebook's TWFE routine is exploratory. Use `scripts/run_did_estimator.sh` and the production pipeline for paper-ready estimates.